In [195]:
import pandas as pd
import numpy as np
import re
import os
import faiss
from sentence_transformers import SentenceTransformer


In [196]:
# Auto-detect or upload dataset.csv
for path in ['dataset.csv', '/content/dataset.csv']:
    if os.path.exists(path):
        df = pd.read_csv(path)
        break
else:
    from google.colab import files
    print('Upload dataset.csv using the picker below:')
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0])


df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7
0,Fungal infection,itching,skin rash,nodal skin eruptions,dischromic patches,NaN,NaN,NaN
1,Allergy,continuous sneezing,shivering,chills,watering from eyes,NaN,NaN,NaN
2,GERD,stomach pain,acidity,ulcers on tongue,vomiting,cough,chest pain,NaN
3,Chronic cholestasis,itching,vomiting,yellowish skin,nausea,loss of appetite,abdominal pain,NaN
4,Drug Reaction,itching,skin rash,stomach pain,burning micturition,spotting urination,NaN,NaN


In [197]:

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

symptom_cols = [c for c in df.columns if c.startswith('Symptom_')]
print(f'Symptom columns found: {symptom_cols}')

def merge_symptoms(row):
    parts = [
        str(row[c]).strip()
        for c in symptom_cols
        if pd.notna(row[c]) and str(row[c]).strip() not in ('', 'nan')
    ]
    return ' '.join(parts)

df['Symptom_Text'] = df.apply(merge_symptoms, axis=1)
df['Cleaned_Text'] = df['Symptom_Text'].apply(clean_text)
df['Question']     = df['Cleaned_Text']
df['Answer']       = df['Disease']

print(f'\n Preprocessing complete — {len(df)} QnA pairs created')
df[['Disease', 'Cleaned_Text']].head(5)

Symptom columns found: ['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7']

 Preprocessing complete — 41 QnA pairs created


,Disease,Cleaned_Text
0,Fungal infection,itching skin rash nodal skin eruptions dischro...
1,Allergy,continuous sneezing shivering chills watering ...
2,GERD,stomach pain acidity ulcers on tongue vomiting...
3,Chronic cholestasis,itching vomiting yellowish skin nausea loss of...
4,Drug Reaction,itching skin rash stomach pain burning micturi...


In [198]:
df.to_csv('cleaned_symptom_data.csv', index=False)

## Step 5 — Embed with Hugging Face MiniLM

Same model as HadithBot: `all-MiniLM-L6-v2`  
**Fix applied:** `.tolist()` instead of `.values` to avoid modality error in newer sentence-transformers.

In [199]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('✅ Model loaded')

texts = df['Cleaned_Text'].tolist()

embeddings = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings, dtype='float32')

print(f'Embeddings shape: {embeddings.shape}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings shape: (41, 384)


In [200]:
np.save('symptom_embeddings.npy', embeddings)

In [201]:
embeddings = np.load('symptom_embeddings.npy')

dimensions  = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimensions)
faiss_index.add(embeddings)

faiss.write_index(faiss_index, 'symptom_faiss.index')
print(f' FAISS index built  — {faiss_index.ntotal} vectors stored')
print(' Saved symptom_faiss.index')

 FAISS index built  — 41 vectors stored
 Saved symptom_faiss.index


In [202]:

faiss_index = faiss.read_index('symptom_faiss.index')
df          = pd.read_csv('cleaned_symptom_data.csv')

def get_similar_diseases(query, top_k=3, model=model, index=faiss_index):
    """
    Given a symptom query string, return the top_k most similar diseases.
    Mirrors HadithBot's get_similar_hadith().
    """
    cleaned         = clean_text(query)

    query_embedding = model.encode([cleaned])
    query_embedding = np.array(query_embedding, dtype='float32')

    distances, indices = index.search(query_embedding, top_k)

    print(f"\n Query : '{query}'")
    print('=' * 60)
    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
        disease  = df['Disease'].iloc[idx]
        symptoms = df['Symptom_Text'].iloc[idx]
        print(f"#{rank}  Disease  : {disease}")
        print(f"    Symptoms : {symptoms}")
        print(f"    Distance : {dist:.4f}")
        print()
        results.append({
            'rank': rank,
            'disease': disease,
            'symptoms': symptoms,
            'distance': round(float(dist), 4)
        })
    return results

In [203]:
get_similar_diseases('high fever headache chills vomiting')


 Query : 'high fever headache chills vomiting'
#1  Disease  : Malaria
    Symptoms : chills vomiting high fever sweating headache nausea diarrhoea
    Distance : 0.2990

#2  Disease  : Typhoid
    Symptoms : chills vomiting fatigue high fever headache nausea constipation
    Distance : 0.3784

#3  Disease  : Dengue
    Symptoms : skin rash chills joint pain vomiting fatigue high fever headache
    Distance : 0.4545



[{'rank': 1,
  'disease': 'Malaria',
  'symptoms': 'chills vomiting high fever sweating headache nausea diarrhoea',
  'distance': 0.299},
 {'rank': 2,
  'disease': 'Typhoid',
  'symptoms': 'chills vomiting fatigue high fever headache nausea constipation',
  'distance': 0.3784},
 {'rank': 3,
  'disease': 'Dengue',
  'symptoms': 'skin rash chills joint pain vomiting fatigue high fever headache',
  'distance': 0.4545}]

In [204]:
get_similar_diseases('itching skin rash fatigue')


 Query : 'itching skin rash fatigue'
#1  Disease  : Chicken pox
    Symptoms : itching skin rash fatigue lethargy high fever headache loss of appetite
    Distance : 0.5382

#2  Disease  : Fungal infection
    Symptoms : itching skin rash nodal skin eruptions dischromic patches
    Distance : 0.5884

#3  Disease  : Drug Reaction
    Symptoms : itching skin rash stomach pain burning micturition spotting urination
    Distance : 0.7107



[{'rank': 1,
  'disease': 'Chicken pox',
  'symptoms': 'itching skin rash fatigue lethargy high fever headache loss of appetite',
  'distance': 0.5382},
 {'rank': 2,
  'disease': 'Fungal infection',
  'symptoms': 'itching skin rash nodal skin eruptions dischromic patches',
  'distance': 0.5884},
 {'rank': 3,
  'disease': 'Drug Reaction',
  'symptoms': 'itching skin rash stomach pain burning micturition spotting urination',
  'distance': 0.7107}]

In [205]:
get_similar_diseases('chest pain breathlessness sweating')


 Query : 'chest pain breathlessness sweating'
#1  Disease  : Heart attack
    Symptoms : vomiting breathlessness sweating chest pain
    Distance : 0.2479

#2  Disease  : Pneumonia
    Symptoms : chills fatigue cough high fever breathlessness sweating malaise
    Distance : 0.6333

#3  Disease  : Tuberculosis
    Symptoms : chills vomiting fatigue weight loss cough high fever breathlessness
    Distance : 0.9062



[{'rank': 1,
  'disease': 'Heart attack',
  'symptoms': 'vomiting breathlessness sweating chest pain',
  'distance': 0.2479},
 {'rank': 2,
  'disease': 'Pneumonia',
  'symptoms': 'chills fatigue cough high fever breathlessness sweating malaise',
  'distance': 0.6333},
 {'rank': 3,
  'disease': 'Tuberculosis',
  'symptoms': 'chills vomiting fatigue weight loss cough high fever breathlessness',
  'distance': 0.9062}]

In [206]:
import os
os.makedirs('templates', exist_ok=True)

with open('templates/index.html', 'w') as f:
    f.write('''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>AI Symptom Checker</title>
  <style>
    *,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
    :root{--blue:#3b82f6;--blue-d:#1d4ed8;--gray-50:#f9fafb;--gray-100:#f3f4f6;--gray-200:#e5e7eb;--gray-400:#9ca3af;--gray-600:#4b5563;--gray-800:#1f2937}
    body{font-family:'Segoe UI',system-ui,sans-serif;background:var(--gray-50);color:var(--gray-800);min-height:100vh}
    header{background:linear-gradient(135deg,#1e40af,#3b82f6);color:#fff;padding:28px 24px;text-align:center}
    header h1{font-size:2rem;font-weight:700}
    header p{margin-top:6px;opacity:.85;font-size:.95rem}
    .badge{display:inline-block;margin-top:12px;background:rgba(255,255,255,.15);border:1px solid rgba(255,255,255,.3);border-radius:99px;padding:4px 14px;font-size:.78rem}
    main{max-width:780px;margin:32px auto;padding:0 16px 60px}
    .card{background:#fff;border-radius:14px;box-shadow:0 1px 4px rgba(0,0,0,.08),0 4px 16px rgba(0,0,0,.06);padding:28px;margin-bottom:24px}
    .card-title{font-size:1.05rem;font-weight:600;margin-bottom:16px;display:flex;align-items:center;gap:8px}
    .search-wrap{position:relative}
    #symptom-input{width:100%;padding:14px 50px 14px 16px;border:2px solid var(--gray-200);border-radius:10px;font-size:1rem;outline:none;transition:border-color .2s}
    #symptom-input:focus{border-color:var(--blue)}
    #add-btn{position:absolute;right:8px;top:50%;transform:translateY(-50%);background:var(--blue);color:#fff;border:none;border-radius:8px;width:38px;height:38px;cursor:pointer;font-size:1.2rem;display:flex;align-items:center;justify-content:center}
    #add-btn:hover{background:var(--blue-d)}
    #suggestions{position:absolute;top:calc(100% + 4px);left:0;right:0;background:#fff;border:1px solid var(--gray-200);border-radius:10px;box-shadow:0 8px 24px rgba(0,0,0,.1);max-height:220px;overflow-y:auto;z-index:50;display:none}
    .sug-item{padding:10px 16px;cursor:pointer;font-size:.92rem}
    .sug-item:hover{background:var(--gray-100)}
    #tags-wrap{display:flex;flex-wrap:wrap;gap:8px;margin-top:14px}
    .tag{display:inline-flex;align-items:center;gap:5px;background:#eff6ff;color:#1d4ed8;border:1px solid #bfdbfe;border-radius:99px;padding:4px 12px;font-size:.83rem;font-weight:500}
    .tag-x{cursor:pointer;opacity:.6;font-size:.9rem}
    .tag-x:hover{opacity:1}
    .actions{display:flex;gap:10px;margin-top:18px}
    .btn{padding:11px 22px;border-radius:8px;border:none;font-size:.92rem;font-weight:600;cursor:pointer;transition:opacity .2s}
    .btn-primary{background:var(--blue);color:#fff;flex:1}
    .btn-primary:hover{opacity:.9}
    .btn-secondary{background:var(--gray-100);color:var(--gray-600)}
    .spinner{display:none;width:22px;height:22px;border:3px solid var(--gray-200);border-top-color:var(--blue);border-radius:50%;animation:spin .7s linear infinite;margin:16px auto}
    @keyframes spin{to{transform:rotate(360deg)}}
    #results-section{display:none}
    .result-card{border:1px solid var(--gray-200);border-radius:12px;padding:18px 20px;margin-bottom:14px}
    .result-header{display:flex;justify-content:space-between;align-items:flex-start;gap:12px}
    .result-rank{width:30px;height:30px;background:var(--blue);color:#fff;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:.8rem;font-weight:700;flex-shrink:0}
    .result-name{font-size:1.1rem;font-weight:700;flex:1}
    .conf-badge{font-size:.8rem;font-weight:600;padding:3px 10px;border-radius:99px}
    .conf-high{background:#dcfce7;color:#166534}
    .conf-med{background:#fef9c3;color:#854d0e}
    .conf-low{background:#fee2e2;color:#991b1b}
    .result-symptoms{margin-top:10px;font-size:.87rem;color:var(--gray-600);line-height:1.6}
    .dist-tag{display:inline-block;margin-top:8px;background:var(--gray-100);color:var(--gray-600);border-radius:6px;padding:2px 8px;font-size:.75rem}
    .disclaimer{background:#fffbeb;border:1px solid #fde68a;border-radius:10px;padding:12px 16px;font-size:.82rem;color:#92400e;margin-top:8px}
    .error-msg{background:#fee2e2;color:#991b1b;border-radius:8px;padding:12px 16px;font-size:.9rem}
    .pipeline{display:grid;grid-template-columns:repeat(auto-fit,minmax(120px,1fr));gap:12px;margin-top:4px}
    .p-step{background:var(--gray-50);border:1px solid var(--gray-200);border-radius:10px;padding:14px 12px;text-align:center}
    .p-step .icon{font-size:1.6rem}
    .p-step .label{margin-top:6px;font-size:.76rem;font-weight:600;color:var(--gray-600)}
  </style>
</head>
<body>
<header>
  <h1>AI Symptom Checker</h1>
  <p>Describe your symptoms and get instant disease matches powered by semantic search</p>
  <span class="badge">MiniLM · FAISS · Flask</span>
</header>
<main>
  <div class="card">
    <div class="card-title">Enter Your Symptoms</div>
    <div class="search-wrap">
      <input id="symptom-input" type="text" placeholder="Type a symptom (e.g. headache, fever)..." autocomplete="off" />
      <button id="add-btn">+</button>
      <div id="suggestions"></div>
    </div>
    <div id="tags-wrap"></div>
    <div class="actions">
      <button class="btn btn-primary" id="analyse-btn">Analyse Symptoms</button>
      <button class="btn btn-secondary" id="clear-btn">Clear</button>
    </div>
  </div>
  <div class="spinner" id="spinner"></div>
  <div id="results-section">
    <div class="card">
      <div class="card-title">Possible Conditions</div>
      <div id="results-body"></div>
      <div class="disclaimer"><strong>Disclaimer:</strong> This tool uses AI similarity search and is for informational purposes only. It is <strong>not a substitute</strong> for professional medical advice.</div>
    </div>
  </div>
  <div class="card">
    <div class="card-title">Pipeline</div>
    <div class="pipeline">
      <div class="p-step"><div class="icon">&#x1f4c4;</div><div class="label">CSV Dataset<br/>41 diseases</div></div>
      <div class="p-step"><div class="icon">&#x1f9f9;</div><div class="label">Text<br/>Preprocessing</div></div>
      <div class="p-step"><div class="icon">&#x1f916;</div><div class="label">MiniLM<br/>Embeddings</div></div>
      <div class="p-step"><div class="icon">&#x1f4e6;</div><div class="label">FAISS<br/>Vector Index</div></div>
      <div class="p-step"><div class="icon">&#x1f50d;</div><div class="label">L2 Similarity<br/>Search</div></div>
      <div class="p-step"><div class="icon">&#x1f310;</div><div class="label">Flask<br/>Web UI</div></div>
    </div>
  </div>
</main>
<script>
  const allSymptoms = {{ symptoms | tojson }};
  const tags = new Set();
  const input      = document.getElementById('symptom-input');
  const suggestBox = document.getElementById('suggestions');
  const tagsWrap   = document.getElementById('tags-wrap');
  const spinner    = document.getElementById('spinner');
  const resultsSection = document.getElementById('results-section');
  const resultsBody    = document.getElementById('results-body');
  input.addEventListener('input', () => {
    const q = input.value.trim().toLowerCase();
    suggestBox.innerHTML = '';
    if (!q) { suggestBox.style.display = 'none'; return; }
    const matches = allSymptoms.filter(s => s.includes(q) && !tags.has(s)).slice(0, 8);
    if (!matches.length) { suggestBox.style.display = 'none'; return; }
    matches.forEach(s => {
      const div = document.createElement('div');
      div.className = 'sug-item'; div.textContent = s;
      div.addEventListener('mousedown', e => { e.preventDefault(); addTag(s); input.value = ''; suggestBox.style.display = 'none'; });
      suggestBox.appendChild(div);
    });
    suggestBox.style.display = 'block';
  });
  input.addEventListener('blur', () => setTimeout(() => suggestBox.style.display = 'none', 150));
  input.addEventListener('keydown', e => { if (e.key === 'Enter') addCurrent(); });
  document.getElementById('add-btn').addEventListener('click', addCurrent);
  function addCurrent() {
    const v = input.value.trim().toLowerCase();
    if (v) { addTag(v); input.value = ''; suggestBox.style.display = 'none'; }
  }
  function addTag(s) {
    if (tags.has(s) || tags.size >= 15) return;
    tags.add(s); renderTags();
  }
  function removeTag(s) { tags.delete(s); renderTags(); }
  function renderTags() {
    tagsWrap.innerHTML = '';
    tags.forEach(s => {
      const span = document.createElement('span');
      span.className = 'tag';
      span.innerHTML = s + ' <span class="tag-x" data-s="' + s + '">x</span>';
      span.querySelector('.tag-x').addEventListener('click', () => removeTag(s));
      tagsWrap.appendChild(span);
    });
  }
  document.getElementById('clear-btn').addEventListener('click', () => {
    tags.clear(); renderTags(); resultsSection.style.display = 'none'; input.value = '';
  });
  document.getElementById('analyse-btn').addEventListener('click', async () => {
    if (!tags.size) { showError('Please add at least one symptom first.'); return; }
    const query = [...tags].join(' ');
    spinner.style.display = 'block'; resultsSection.style.display = 'none';
    try {
      const res = await fetch('/search', { method: 'POST', headers: {'Content-Type':'application/json'}, body: JSON.stringify({query}) });
      const data = await res.json();
      spinner.style.display = 'none';
      if (data.error) { showError(data.error); return; }
      renderResults(data.results);
    } catch(e) { spinner.style.display = 'none'; showError('Server error.'); }
  });
  function renderResults(results) {
    resultsBody.innerHTML = '';
    if (!results || !results.length) { resultsBody.innerHTML = '<p style="text-align:center;color:#9ca3af;padding:32px 0">No results found.</p>'; resultsSection.style.display='block'; return; }
    results.forEach((r, i) => {
      const cls = r.confidence >= 60 ? 'conf-high' : r.confidence >= 35 ? 'conf-med' : 'conf-low';
      const div = document.createElement('div'); div.className = 'result-card';
      div.innerHTML = '<div class="result-header"><div class="result-rank">' + (i+1) + '</div><div class="result-name">' + r.disease + '</div><span class="conf-badge ' + cls + '">' + r.confidence + '% match</span></div><div class="result-symptoms"><strong>Known symptoms:</strong> ' + r.symptoms + '</div><span class="dist-tag">L2 distance: ' + r.distance + '</span>';
      resultsBody.appendChild(div);
    });
    resultsSection.style.display = 'block';
    resultsSection.scrollIntoView({behavior:'smooth', block:'start'});
  }
  function showError(msg) {
    resultsBody.innerHTML = '<div class="error-msg">' + msg + '</div>';
    resultsSection.style.display = 'block';
  }
</script>
</body>
</html>''')



Run the cell below. A **public URL** will appear open it in any browser.

In [ ]:

import subprocess, sys, time, re

subprocess.run(['npm', 'install', '-g', 'localtunnel'], capture_output=True)

print('Starting Flask...')
flask_proc = subprocess.Popen(
    [sys.executable, 'app.py'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)
print('Flask running on http://127.0.0.1:5000')

result = subprocess.run(['curl', '-s', 'https://ipv4.icanhazip.com'],
                        capture_output=True, text=True)
external_ip = result.stdout.strip()
print(f'\n Your password for the tunnel page: {external_ip}\n')
tunnel = subprocess.Popen(
    ['lt', '--port', '5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in tunnel.stdout:
    if 'https://' in line:
        url = re.search(r'https://[\w.-]+', line).group()

        print(f'   Open this URL:\n\n     {url}')
        print(f'\n  Enter this as password:\n\n     {external_ip}')
        print('\n  Keep this cell running so the app would work.')
        break